In [1]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U accelerate
!pip install trl
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 15.8 MB/s eta 0:00:00


In [ ]:
import os
import json
import warnings
import torch 
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, PeftModel

from huggingface_hub import login

from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
os.chdir(base_path)

from dotenv import load_dotenv
load_dotenv(override=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


True

In [7]:
hf_token = os.environ.get("HUGGINGFACE_KEY")
login(token=hf_token)

In [12]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16 
)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [ ]:
peft_config = LoraConfig(
    r=16,              
    lora_alpha=32,    
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,   
    bias="none",
    task_type="CAUSAL_LM",
)

with open(os.path.join("data", "gemma_finetune.json"), 'r', encoding='utf-8') as file:
    data_list = json.load(file)

texts = [
    f"<start_of_turn>user\n{item['instruction']}\n\nInput: {item['input']}<end_of_turn>\n<start_of_turn>model\n{item['output']}<end_of_turn>"
    for item in data_list
]

In [15]:
dataset = Dataset.from_dict({"text": texts})

training_args = TrainingArguments(
    output_dir="./gemma-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=50,
    logging_steps=10,
    fp16=False,
    bf16=True,
    remove_unused_columns=False 
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,2.043204
20,0.465327
30,0.341650
40,0.304070
50,0.290070


('gemma-finetuned-final/tokenizer_config.json',
 'gemma-finetuned-final/tokenizer.json')

In [16]:
trainer.train()

new_model_path = "gemma-finetuned-final"

trainer.model.save_pretrained(new_model_path)
tokenizer.save_pretrained(new_model_path)

KeyboardInterrupt: 